In [1]:
from pathlib import Path
from typing import List, Tuple, Dict
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans

import torch
from transformers import (
    AutoTokenizer,
        AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from numpy.linalg import norm

# Configuração e Verificação Inicial

In [3]:
EXPORT_DIR = "geocorpus"  # mude o nome se quiser
Path(EXPORT_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(EXPORT_DIR)

In [4]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)

JSON_PATH = "../../data/geocorpus-v2.json"
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

with open(JSON_PATH, encoding="utf-8") as f:
    raw = json.load(f)

In [5]:
records_geo = [
    {
        "sentence_id": i,
        "tokens": item["tokens"],
        "ner_tags": item["ner_tokens"],
    }
    for i, item in enumerate(raw)
]

geocorpus_full = Dataset.from_list(records_geo)

In [6]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geocorpus_full["ner_tags"] for l in sent})
label2id   = {l: i for i, l in enumerate(label_list)}
id2label   = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [7]:
id2label

{0: 'B-ambienteSedimentacao',
 1: 'B-baciaSedimentar',
 2: 'B-bentonico',
 3: 'B-campoPetrolifero',
 4: 'B-constituinteRochaSedimentar',
 5: 'B-contextoGeologicoDeBacia',
 6: 'B-elementoQuimico',
 7: 'B-eon',
 8: 'B-epoca',
 9: 'B-era',
 10: 'B-estratigrafia',
 11: 'B-estruturaGeologica',
 12: 'B-estruturaSedimentar',
 13: 'B-fosseis',
 14: 'B-geomorfologia',
 15: 'B-granulometria',
 16: 'B-idade',
 17: 'B-magmaticas',
 18: 'B-metamorficas',
 19: 'B-mineral',
 20: 'B-periodo',
 21: 'B-planctonico',
 22: 'B-procedimentoMetodologico',
 23: 'B-sedimentaresCarbonaticas',
 24: 'B-sedimentaresOrganicas',
 25: 'B-sedimentaresQuimicas',
 26: 'B-sedimentaresSiliciclasticas',
 27: 'B-sistemaPetrolifero',
 28: 'B-unidadeEstratigrafica',
 29: 'B-unidadeGeotectonica',
 30: 'I-ambienteSedimentacao',
 31: 'I-baciaSedimentar',
 32: 'I-bentonico',
 33: 'I-campoPetrolifero',
 34: 'I-constituinteRochaSedimentar',
 35: 'I-contextoGeologicoDeBacia',
 36: 'I-epoca',
 37: 'I-era',
 38: 'I-estratigrafia',
 39

In [8]:
NUM_LABELS

57

# Splits

In [9]:
def split_standard(ds: Dataset) -> DatasetDict:
    """Usa coluna trainingTest do CSV (80/20 original)."""
    if "trainingTest" not in df.columns:
        raise ValueError("CSV não contém a coluna 'trainingTest'")
    train_ids = df.loc[df["trainingTest"] == "training", SENT_COL].unique()
    test_ids = df.loc[df["trainingTest"] == "test", SENT_COL].unique()
    return DatasetDict(
        train=ds.filter(lambda ex: ex["sentence_id"] in train_ids),
        dev=ds.filter(lambda ex: ex["sentence_id"] in test_ids),
    )

In [10]:
def random_splits(
    ds: Dataset, test_size=0.2, seeds: List[int] = range(30)
) -> List[DatasetDict]:
    triples = []
    for s in seeds:
        train, dev = ds.train_test_split(test_size=test_size, seed=s).values()
        triples.append(DatasetDict(train=train, dev=dev))
    return triples

In [11]:
# def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
#     lengths = np.array([len(t) for t in ds["tokens"]])
#     thr = np.percentile(lengths, 100 * (1 - top_pct))
#     mask = lengths >= thr
#     return DatasetDict(train=ds.filter(~mask), dev=ds.filter(mask))


def split_heur_length(ds: Dataset, top_pct: float = 0.20) -> DatasetDict:
    """20 % das sentenças mais longas viram conjunto de validação (dev)."""
    lengths = np.array([len(t) for t in ds["tokens"]])
    thr = np.percentile(lengths, 100 * (1 - top_pct))  
    mask = lengths >= thr  

    dev_idx = np.where(mask)[0].tolist()  # índices → list[int]
    train_idx = np.where(~mask)[0].tolist()

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Tamanho da sentenças


# def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
#     freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
#     rare = {w for w, c in freq.items() if c <= freq_thr}

#     def has_rare(example):
#         return any(w.lower() in rare for w in example["tokens"])

#     return DatasetDict(
#         train=ds.filter(lambda ex: not has_rare(ex)), dev=ds.filter(has_rare)
#     )


def split_heur_rare(ds: Dataset, freq_thr: int = 5) -> DatasetDict:
    freq = Counter(w.lower() for sent in ds["tokens"] for w in sent)
    rare = {w for w, c in freq.items() if c <= freq_thr}

    keep_dev = []
    for sent in ds["tokens"]:
        print(sent)
        keep_dev.append(any(w.lower() in rare for w in sent))

    dev_idx = [i for i, x in enumerate(keep_dev) if x]
    train_idx = [i for i, x in enumerate(keep_dev) if not x]

    return DatasetDict(
        train=ds.select(train_idx),
        dev=ds.select(dev_idx),
    )

    # Raridade dos tokens

In [12]:
def split_adversarial(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeds = model.encode([" ".join(t) for t in ds["tokens"]], show_progress_bar=False)
    tree = BallTree(embeds, leaf_size=40)

    idx_train, idx_test = set(range(len(ds))), []
    # semente = ponto mais central
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    idx_train.remove(seed_idx)
    idx_test.append(seed_idx)
    print(k)
    while len(idx_test) < k:
        print(len(idx_test))
        dists, _ = tree.query(embeds[list(idx_train)], k=1, return_distance=True)
        nxt = list(idx_train)[int(np.argmax(dists))]
        idx_train.remove(nxt)
        idx_test.append(nxt)

    return DatasetDict(
        train=ds.select(sorted(idx_train)),
        dev=ds.select(sorted(idx_test)),
    )

    # Maximizando Wassertein Distance


def split_adversarial_fast(ds: Dataset, pct_test: float = 0.20) -> DatasetDict:
    """
    Farthest-Point Sampling aproximando Wasserstein – versão vetorizada.
    Seleciona pct_test (~20 %) das sentenças como conjunto 'dev'.
    """
    k = int(len(ds) * pct_test)
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

    embeds = model.encode(
        [" ".join(t) for t in ds["tokens"]],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,  # acelera distância euclidiana ≈ cos
    )

    n = embeds.shape[0]
    idx_all = np.arange(n)

    # 1) ponto mais "central" (norma mais distante da média)
    seed_idx = np.argmax(np.linalg.norm(embeds - embeds.mean(0), axis=1))
    selected = [seed_idx]

    # 2) vetor de distâncias mínimas a qualquer ponto já escolhido
    min_dists = np.linalg.norm(embeds - embeds[seed_idx], axis=1)

    while len(selected) < k:
        next_idx = np.argmax(min_dists)
        selected.append(next_idx)

        # atualiza min_dists com a distância ao novo ponto — tudo de uma vez
        d_new = np.linalg.norm(embeds - embeds[next_idx], axis=1)
        min_dists = np.minimum(min_dists, d_new)

    train_idx = np.setdiff1d(idx_all, selected, assume_unique=True)

    return DatasetDict(
        train=ds.select(train_idx.tolist()),
        dev=ds.select(selected),
    )

In [13]:
# def loc_split(
#     dataset: Dataset, pct_test: float = 0.20, ngram: int = 4, seed: int = 42
# ) -> DatasetDict:
#     """
#     Split baseado em baixa sobreposição léxica (4-gram Jaccard).
#     Teste = pct_test das sentenças com menor overlap em relação ao pool.
#     """
#     # 1. Texto plano por sentença
#     docs = [" ".join(toks) for toks in dataset["tokens"]]

#     # 2. Vetorizar 4-grams (binário)
#     vect = CountVectorizer(
#         analyzer="word", ngram_range=(ngram, ngram), binary=True
#     ).fit(docs)
#     X = vect.transform(docs)

#     # 3. Similaridade Jaccard aproximada com matriz binária
#     # Jaccard(A,B) = |A∩B|/|A∪B| = 1 - |AΔB|/|A∪B|
#     # Usamos: overlap = (A·Bᵀ) / (|A|+|B|-A·Bᵀ)
#     bin_counts = X.sum(axis=1).A1

#     # Para cada doc i, escolhemos vizinho + próximo (fast):
#     from sklearn.metrics.pairwise import cosine_similarity

#     # (cosine no binário ∝ |A∩B|)
#     sim = cosine_similarity(X, dense_output=False)
#     # Soma dos top-k overlaps (k=5) como score
#     k = 5
#     topk = np.zeros(len(dataset))
#     for i in range(sim.shape[0]):
#         row = sim.getrow(i).toarray()[0]
#         idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
#         # overlap ≈ |∩|
#         inter = row[idx] * bin_counts[i]
#         uni = bin_counts[i] + bin_counts[idx] - inter
#         topk[i] = (inter / uni).mean()

#     # 4. Ordenar por overlap crescente ⇒ mais “novos” vão p/ teste
#     order = np.argsort(topk)
#     n_test = int(len(dataset) * pct_test)
#     test_idx = order[:n_test]
#     train_idx = order[n_test:]

#     return DatasetDict(
#         {"train": dataset.select(train_idx), "test": dataset.select(test_idx)}
#     )

In [14]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [16]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [17]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    while len(test_idx) < int(pct_test*len(dataset)):
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [20]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [22]:
standard_split = std_split(geocorpus_full)
print('std')
# random_splt = random_splits(geocorpus_full)
# print('random')
heur_len = heur_len_split(geocorpus_full)
print("heur_len")
heur_rare = heur_rare_split(geocorpus_full)
print("heur_rare")
advers = adversarial_split(geocorpus_full)
print("advs")
loc = loc_split(geocorpus_full)
print("loc")
semantic = semantic_cluster_split(geocorpus_full)
print("semantic")
reverse = reverse_curriculum_split(geocorpus_full)
print("reverse")

std
heur_len
heur_rare
Selecionando 1054 sentenças para teste…
  0 selecionadas…
  24 selecionadas…
  47 selecionadas…
  71 selecionadas…
  96 selecionadas…
  121 selecionadas…
  144 selecionadas…
  165 selecionadas…
  187 selecionadas…
  211 selecionadas…
  232 selecionadas…
  251 selecionadas…
  271 selecionadas…
  288 selecionadas…
  305 selecionadas…
  329 selecionadas…
  349 selecionadas…
  364 selecionadas…
  383 selecionadas…
  399 selecionadas…
  417 selecionadas…
  433 selecionadas…
  452 selecionadas…
  470 selecionadas…
  488 selecionadas…
  501 selecionadas…
  515 selecionadas…
  529 selecionadas…
  542 selecionadas…
  554 selecionadas…
  565 selecionadas…
  577 selecionadas…
  590 selecionadas…
  597 selecionadas…
  610 selecionadas…
  628 selecionadas…
  642 selecionadas…
  658 selecionadas…
  671 selecionadas…
  685 selecionadas…
  698 selecionadas…
  713 selecionadas…
  724 selecionadas…
  738 selecionadas…
  754 selecionadas…
  771 selecionadas…
  785 selecionadas…
  7

100%|██████████| 5272/5272 [00:00<00:00, 1343808.61it/s]


semantic
reverse


In [23]:
standard_split

DatasetDict({
    train: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 4217
    })
    val: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 527
    })
    test: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 528
    })
})

In [24]:
heur_len

DatasetDict({
    train: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 3691
    })
    val: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 527
    })
    test: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 1054
    })
})

In [25]:
heur_rare

DatasetDict({
    train: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 3691
    })
    val: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 527
    })
    test: Dataset({
        features: ['sentence_id', 'tokens', 'ner_tags'],
        num_rows: 1054
    })
})

In [26]:
from collections import Counter, defaultdict
from typing import Dict, List, Tuple
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances

In [27]:
def _label_names(ds_split):
    """Tenta obter nomes dos rótulos a partir do schema do HF Datasets."""
    feat = ds_split.features["ner_tags"]
    if hasattr(feat, "feature") and hasattr(feat.feature, "names"):
        return list(feat.feature.names)
    # fallback: cria nomes genéricos a partir dos valores observados
    vals = set()
    for ex in ds_split:
        vals.update(ex["ner_tags"])
    return list(vals)

def _flatten_labels(ds_split):
    for ex in ds_split:
        for y in ex["ner_tags"]:
            yield y

def _sent_texts(ds_split):
    # junta tokens com espaço para TF-IDF
    for ex in ds_split:
        yield " ".join(ex["tokens"])

def _token_stream(ds_split):
    for ex in ds_split:
        for t in ex["tokens"]:
            yield t

def _entity_counts(ds_split, id2name):
    # conta rótulos (token-level) e também sequência de entidades (simples: conta B-*)
    c_tok = Counter()
    for ex in ds_split:
        for y in ex["ner_tags"]:
             c_tok[y] += 1
    return c_tok

def _lengths(ds_split):
    return np.array([len(ex["tokens"]) for ex in ds_split], dtype=int)

def _oov_rate(train_vocab: set, ds_split) -> float:
    tot = 0
    oov = 0
    for ex in ds_split:
        for t in ex["tokens"]:
            tot += 1
            if t not in train_vocab:
                oov += 1
    return oov / max(tot, 1)

def _tfidf_centroid(texts: List[str], vectorizer: TfidfVectorizer=None):
    # retorna (centro, vectorizer)
    fit_new = vectorizer is None
    if fit_new:
        vectorizer = TfidfVectorizer(min_df=2, max_df=0.98, ngram_range=(1,2))
        X = vectorizer.fit_transform(texts)
    else:
        X = vectorizer.transform(texts)
    # média das linhas (TF-IDF já vem L2-normalizado por linha)
    centroid = X.mean(axis=0)                       # <- vira np.matrix
    centroid = np.asarray(centroid).ravel()         # -> ndarray (D,)
    # (opcional) normalizar o centróide para norma 1, por simetria do cosseno
    denom = norm(centroid)
    if denom > 0:
        centroid = centroid / denom
    centroid = centroid.reshape(1, -1)              # -> (1, D) para pairwise
    return centroid, vectorizer

def _cos(a, b):
    """
    Distância do cosseno com entrada tolerante (matrix/array/1D/2D).
    Retorna float.
    """
    a = np.asarray(a)
    b = np.asarray(b)
    if a.ndim == 1:
        a = a.reshape(1, -1)
    if b.ndim == 1:
        b = b.reshape(1, -1)
    return float(cosine_distances(a, b)[0, 0])

In [28]:
def analyze_instances_for_insights(ds_dict, top_k_tokens=20):
    id2name = _label_names(ds_dict["train"])

    # comprimentos
    lens = {p: np.array([len(ex["tokens"]) for ex in ds_dict[p]], int)
            for p in ["train","val","test"]}
    q = {p: tuple(map(int, np.quantile(lens[p], [0.1, 0.5, 0.9, 0.99])))
         for p in lens}

    # OOV: val/test vs vocabulário do treino
    train_vocab = set(_token_stream(ds_dict["train"]))
    oov_val  = _oov_rate(train_vocab, ds_dict["val"])
    oov_test = _oov_rate(train_vocab, ds_dict["test"])

    # rótulos raros (no conjunto completo)
    total_counts = Counter()
    for p in ["train","val","test"]:
        total_counts.update(_entity_counts(ds_dict[p], id2name))
    rares = sorted(total_counts.items(), key=lambda kv: kv[1])[:5]

    # tokens mais característicos (teste vs treino) por log-odds simples
    def _log_odds(alpha=0.01):
        c_tr = Counter(_token_stream(ds_dict["train"]))
        c_te = Counter(_token_stream(ds_dict["test"]))
        V = set(c_tr) | set(c_te)
        n_tr, n_te = sum(c_tr.values()), sum(c_te.values())
        scores = [(w, np.log((c_te[w]+alpha)/(n_te+alpha*len(V))) -
                      np.log((c_tr[w]+alpha)/(n_tr+alpha*len(V)))) for w in V]
        scores.sort(key=lambda x: x[1], reverse=True)
        return [w for w,_ in scores[:top_k_tokens]], [w for w,_ in scores[-top_k_tokens:]]

    top_test, top_train = _log_odds()

    return {
        "length_quantiles": q,  # p10, p50, p90, p99 por split
        "oov_rate_val_vs_train": oov_val,
        "oov_rate_test_vs_train": oov_test,
        "5_rare_labels_overall_(label,count)": rares,
        "tokens_more_characteristic_in_test": top_test,
        "tokens_more_characteristic_in_train": top_train,
    }

In [29]:
def save_split_insights_to_csv(split_name: str, ins: dict, out_root: str = "insights_out") -> None:
    """
    Salva cada parte de `ins` (dict) em um CSV distinto, dentro de insights_out/<split_name>/.
    Estruturas:
      - length_quantiles.csv: linhas longas (part, quantile, value)
      - oov_rates.csv       : (metric, value)
      - rare_labels.csv     : (label, count) [top-5]
      - tokens_test.csv     : (rank, token)
      - tokens_train.csv    : (rank, token)
    """
    out_dir = os.path.join(out_root, split_name)
    os.makedirs(out_dir, exist_ok=True)

    # 1) Quantis de comprimento (formato longo)
    # ins["length_quantiles"] = { 'train': (p10,p50,p90,p99), 'val': ..., 'test': ... }
    rows = []
    for part, quads in ins["length_quantiles"].items():
        p10, p50, p90, p99 = quads
        rows.extend([
            {"part": part, "quantile": "p10", "value": int(p10)},
            {"part": part, "quantile": "p50", "value": int(p50)},
            {"part": part, "quantile": "p90", "value": int(p90)},
            {"part": part, "quantile": "p99", "value": int(p99)},
        ])
    pd.DataFrame(rows).to_csv(os.path.join(out_dir, "length_quantiles.csv"), index=False)

    # 2) OOV val/test vs train
    pd.DataFrame(
        [
            {"metric": "oov_rate_val_vs_train",  "value": float(ins["oov_rate_val_vs_train"])},
            {"metric": "oov_rate_test_vs_train", "value": float(ins["oov_rate_test_vs_train"])},
        ]
    ).to_csv(os.path.join(out_dir, "oov_rates.csv"), index=False)

    # 3) Rótulos raros (top-5)
    # chave tem um nome longo; localiza pela substring "rare_labels"
    rare_key = next(k for k in ins.keys() if "rare_labels" in k)
    # ins[rare_key] = [(label, count), ...]
    pd.DataFrame(ins[rare_key], columns=["label", "count"]).to_csv(
        os.path.join(out_dir, "rare_labels.csv"), index=False
    )

    # 4) Tokens mais característicos (teste vs treino)
    pd.DataFrame(
        {
            "rank": list(range(1, len(ins["tokens_more_characteristic_in_test"]) + 1)),
            "token": ins["tokens_more_characteristic_in_test"],
        }
    ).to_csv(os.path.join(out_dir, "tokens_test.csv"), index=False)

    pd.DataFrame(
        {
            "rank": list(range(1, len(ins["tokens_more_characteristic_in_train"]) + 1)),
            "token": ins["tokens_more_characteristic_in_train"],
        }
    ).to_csv(os.path.join(out_dir, "tokens_train.csv"), index=False)

In [30]:
splits = {
    "standard": standard_split,
    "heur_len": heur_len,
    "heur_rare": heur_rare,
    "adversarial": advers,
    "loc": loc,
    "semantic": semantic,
    "reverse": reverse,
}

# 1) Insights por split
insights = {}
for name, ds_dict in splits.items():
    ins = analyze_instances_for_insights(ds_dict, top_k_tokens=20)
    insights[name] = ins
    print(f"\n[{name}]")
    print(ins)
    save_split_insights_to_csv(name, ins, out_root="insights_out")


[standard]
{'length_quantiles': {'train': (13, 28, 53, 99), 'val': (14, 29, 52, 97), 'test': (14, 28, 54, 100)}, 'oov_rate_val_vs_train': 0.05799260936941233, 'oov_rate_test_vs_train': 0.059622686283815854, '5_rare_labels_overall_(label,count)': [('I-mineral', 2), ('B-campoPetrolifero', 6), ('I-campoPetrolifero', 6), ('I-procedimentoMetodologico', 7), ('I-bentonico', 8)], 'tokens_more_characteristic_in_test': ['cuiú', 'entradas', 'milho', 'jaibaras', '100m', 'ursa', 'haploides', 'orogênese', 'ronda', 'controlados', 'biológicas', 'face', 'talco', 'micritizados', '96', 'diminuta', 'www', 'génese', 'h2s', 'cma-rtp-uw1000m'], 'tokens_more_characteristic_in_train': ['silúrico', 'carvão', 'kunguriano', 'exposição', '+', 'brasília', 'antepaís', 'quase', 'profundas', 'eram', 'álcali-feldspato', 'subsistema', 'responsável', 'europa', 'sobradinho', 'próximo', 'termos', 'sete', 'chance', 'análises']}

[heur_len]
{'length_quantiles': {'train': (13, 28, 52, 99), 'val': (14, 28, 53, 92), 'test': (1

In [34]:
def class_distribution_table(ds_dict) -> Tuple[pd.DataFrame, Dict[str, np.ndarray]]:
    id2name_train = _label_names(ds_dict["train"])

    # 1) Contagens por part e união de labels observadas em todo o DatasetDict
    counts_by_part = {}
    labels_union = set(id2name_train)
    for part in ["train", "val", "test"]:
        c = _entity_counts(ds_dict[part], id2name_train)  # mantém sua assinatura atual
        counts_by_part[part] = c
        labels_union |= set(c.keys())

    # 2) Ordem estável: primeiro as labels do treino, depois as novas (ordenadas)
    new_labels = sorted(labels_union - set(id2name_train))
    labels_all = list(id2name_train) + new_labels

    name2id = {n: i for i, n in enumerate(labels_all)}
    n_labels = len(labels_all)

    # 3) Monta DF longo + vetores normalizados por split
    rows = []
    label_vecs = {}
    for part in ["train", "val", "test"]:
        c = counts_by_part[part]
        total = int(sum(c.values()))
        vec = np.zeros((1, n_labels), dtype=float)

        # preenche vetor e linhas do DF na ordem labels_all (inclui zero para rótulos ausentes)
        for lab in labels_all:
            cnt = int(c.get(lab, 0))
            vec[0, name2id[lab]] = cnt
            rows.append((part, lab, cnt, cnt / max(total, 1)))

        # normaliza pelo total do part (evita div by zero)
        label_vecs[part] = vec / max(total, 1)

    class_df = pd.DataFrame(rows, columns=["split", "label", "count", "prop"])
    return class_df, label_vecs

In [32]:
def save_class_distribution_to_csv(split_name: str,
                                   class_df: pd.DataFrame,
                                   label_vecs: dict,
                                   out_root: str = "class_dist_out") -> None:
    # cria pasta por split
    out_dir = os.path.join(out_root, split_name)
    os.makedirs(out_dir, exist_ok=True)

    # 1) DF longo original
    class_df.to_csv(os.path.join(out_dir, "class_distribution_long.csv"), index=False)

    # 2) Tabelas dinâmicas (counts e props)
    counts = class_df.pivot(index="label", columns="split", values="count").fillna(0).astype(int)
    props  = class_df.pivot(index="label", columns="split", values="prop").fillna(0.0)

    counts.to_csv(os.path.join(out_dir, "counts_pivot.csv"))
    props.to_csv(os.path.join(out_dir, "props_pivot.csv"))

    # 3) label_vecs: normalizado por split (cada linha = um split; colunas = rótulos)
    # label_vecs[part] = array shape (1, n_labels); precisamos do nome das colunas na ordem correta
    # recupera a ordem das labels do DF original
    labels_order = (
        class_df[["label"]]
        .drop_duplicates()["label"]
        .tolist()
    )
    rows = []
    for part, vec in label_vecs.items():
        row = {"split": part}
        for i, lab in enumerate(labels_order):
            row[lab] = float(vec[0, i]) if vec.ndim == 2 else float(vec[i])
        rows.append(row)
    lv_df = pd.DataFrame(rows).set_index("split")
    lv_df = lv_df[labels_order]  # garante ordem das colunas
    lv_df.to_csv(os.path.join(out_dir, "label_vecs.csv"))

    # 4) também salva um CSV por part (opcional, útil para comparar rápido)
    for part in label_vecs:
        pd.DataFrame([lv_df.loc[part]], index=[part]).to_csv(
            os.path.join(out_dir, f"label_vecs_{part}.csv")
        )

In [35]:
# =========================================================
# 2) Análise quantitativa das classes (train/val/test)
# =========================================================
class_distribution = {}
for name, ds_dict in splits.items():
    df_long, vecs = class_distribution_table(ds_dict)
    class_distribution[name] = df_long
    print(f"\n[{name}]")
    print(df_long.head())  # evita imprimir tudo
    save_class_distribution_to_csv(name, df_long, vecs, out_root="class_dist_out")


[standard]
   split                       label  count      prop
0  train      B-ambienteSedimentacao    112  0.000843
1  train  I-sedimentaresCarbonaticas     77  0.000580
2  train      B-sedimentaresQuimicas     10  0.000075
3  train  B-sedimentaresCarbonaticas    293  0.002206
4  train                       B-era    323  0.002432

[heur_len]
   split                       label  count      prop
0  train      B-ambienteSedimentacao     94  0.000808
1  train  I-sedimentaresCarbonaticas     67  0.000576
2  train      B-sedimentaresQuimicas     10  0.000086
3  train  B-sedimentaresCarbonaticas    256  0.002200
4  train                       B-era    291  0.002501

[heur_rare]
   split                       label  count      prop
0  train      B-ambienteSedimentacao    101  0.000894
1  train  I-sedimentaresCarbonaticas     65  0.000575
2  train      B-sedimentaresQuimicas      8  0.000071
3  train  B-sedimentaresCarbonaticas    248  0.002195
4  train             I-estratigrafia     22  

In [36]:
def cosine_within_split(ds_dict) -> pd.DataFrame:
    # palavras (TF-IDF) — um vocabulário por split já é suficiente aqui
    tr_texts = list(_sent_texts(ds_dict["train"]))
    va_texts = list(_sent_texts(ds_dict["val"]))
    te_texts = list(_sent_texts(ds_dict["test"]))
    cen_tr, vec = _tfidf_centroid(tr_texts, None)
    cen_va, _   = _tfidf_centroid(va_texts, vec)
    cen_te, _   = _tfidf_centroid(te_texts, vec)

    # rótulos
    _, label_vecs = class_distribution_table(ds_dict)

    rows = [
        ["labels","train","val",  _cos(label_vecs["train"], label_vecs["val"])],
        ["labels","train","test", _cos(label_vecs["train"], label_vecs["test"])],
        ["labels","val","test",   _cos(label_vecs["val"],   label_vecs["test"])],
        ["words","train","val",   _cos(cen_tr, cen_va)],
        ["words","train","test",  _cos(cen_tr, cen_te)],
        ["words","val","test",    _cos(cen_va, cen_te)],
    ]
    return pd.DataFrame(rows, columns=["space","a","b","cosine_distance"])

In [37]:

def save_cosine_to_csv(split_name: str,
                       cos_df: pd.DataFrame,
                       out_root: str = "cosine_out") -> None:
    # cria pasta do split
    out_dir = os.path.join(out_root, split_name)
    os.makedirs(out_dir, exist_ok=True)

    # 1) DF completo
    cos_df.to_csv(os.path.join(out_dir, "cosine_all.csv"), index=False)

    # 2) Subconjuntos por espaço
    df_labels = cos_df[cos_df["space"] == "labels"].reset_index(drop=True)
    df_words  = cos_df[cos_df["space"] == "words"].reset_index(drop=True)

    df_labels.to_csv(os.path.join(out_dir, "cosine_labels.csv"), index=False)
    df_words.to_csv(os.path.join(out_dir, "cosine_words.csv"), index=False)

    # 3) Matrizes em formato largo (úteis para comparação rápida)
    # cada célula representa o cos(a,b)
    def _to_matrix(df_space: pd.DataFrame) -> pd.DataFrame:
        # cria coluna "pair" = "a_b" para pivot
        tmp = df_space.assign(pair=df_space["a"] + "_" + df_space["b"])
        mat = tmp.pivot_table(index="space", columns="pair",
                              values="cosine_distance", aggfunc="first")
        # A linha é o próprio "space" (labels/words); tiramos o índice para salvar “flat”
        mat.reset_index(drop=True, inplace=True)
        return mat

    _to_matrix(df_labels).to_csv(os.path.join(out_dir, "matrix_labels.csv"), index=False)
    _to_matrix(df_words).to_csv(os.path.join(out_dir, "matrix_words.csv"), index=False)


In [38]:
# =========================================================
# 3) Distância do cosseno entre train/val/test (por split)
#    (labels e palavras/TF-IDF)
# =========================================================
coisine = {}
all_rows = []  # para o resumo de todos os splits
for name, ds_dict in splits.items():
    df = cosine_within_split(ds_dict)
    coisine[name] = df
    print(f"\n[{name}]")
    print(df)
    save_cosine_to_csv(name, df, out_root="cosine_out")

    # anexa nome do split para o resumo agregado
    df_with_split = df.copy()
    df_with_split.insert(0, "split", name)
    all_rows.append(df_with_split)


[standard]
    space      a     b  cosine_distance
0  labels  train   val         0.000005
1  labels  train  test         0.000007
2  labels    val  test         0.000014
3   words  train   val         0.067678
4   words  train  test         0.068365
5   words    val  test         0.114670

[heur_len]
    space      a     b  cosine_distance
0  labels  train   val         0.000005
1  labels  train  test         0.000002
2  labels    val  test         0.000004
3   words  train   val         0.063732
4   words  train  test         0.040181
5   words    val  test         0.081791

[heur_rare]
    space      a     b  cosine_distance
0  labels  train   val         0.000005
1  labels  train  test         0.000004
2  labels    val  test         0.000008
3   words  train   val         0.068075
4   words  train  test         0.046645
5   words    val  test         0.083270

[adversarial]
    space      a     b  cosine_distance
0  labels  train   val         0.000008
1  labels  train  test      

In [39]:
summary = pd.concat(all_rows, ignore_index=True)
os.makedirs("cosine_out", exist_ok=True)
summary.to_csv(os.path.join("cosine_out", "cosine_summary_all_splits.csv"), index=False)

In [40]:
# =========================================================
def cosine_between_splits(splits: Dict[str, "DatasetDict"]) -> Dict[str, pd.DataFrame]:
    names = list(splits.keys())

    # --- (A) preparar centróides TF-IDF com VOCABULÁRIO COMPARTILHADO
    # fit no conjunto de todos os trains para comparabilidade direta
    all_train_texts = []
    for name in names:
        all_train_texts.extend(_sent_texts(splits[name]["train"]))
    shared_vec = TfidfVectorizer(min_df=2, max_df=0.98, ngram_range=(1,2)).fit(list(all_train_texts))

    word_centers_train = {}
    word_centers_test  = {}
    for name in names:
        cen_tr, _ = _tfidf_centroid(list(_sent_texts(splits[name]["train"])), shared_vec)
        cen_te, _ = _tfidf_centroid(list(_sent_texts(splits[name]["test"])),  shared_vec)
        word_centers_train[name] = cen_tr
        word_centers_test[name]  = cen_te

    # --- (B) preparar vetores de rótulos normalizados
    label_vecs_train = {}
    label_vecs_test  = {}
    for name in names:
        _, lv = class_distribution_table(splits[name])
        label_vecs_train[name] = lv["train"]
        label_vecs_test[name]  = lv["test"]

    # --- (C) montar matrizes de distâncias (palavras e rótulos, train/test)
    def _pairwise(names, get_vec):
        M = np.zeros((len(names), len(names)))
        for i,a in enumerate(names):
            for j,b in enumerate(names):
                M[i,j] = _cos(get_vec(a), get_vec(b))
        return pd.DataFrame(M, index=names, columns=names)

    cos_train_words = _pairwise(names, lambda n: word_centers_train[n])
    cos_test_words  = _pairwise(names, lambda n: word_centers_test[n])
    cos_train_lbls  = _pairwise(names, lambda n: label_vecs_train[n])
    cos_test_lbls   = _pairwise(names, lambda n: label_vecs_test[n])

    return {
        "cos_train_words": cos_train_words,
        "cos_test_words":  cos_test_words,
        "cos_train_labels": cos_train_lbls,
        "cos_test_labels":  cos_test_lbls,
    }

In [41]:
between = cosine_between_splits(splits)

In [42]:
out_root = "cosine_between_out"
os.makedirs(out_root, exist_ok=True)

# 1) Salva as 4 matrizes completas
file_map = {
    "cos_train_words":  "cos_train_words.csv",
    "cos_test_words":   "cos_test_words.csv",
    "cos_train_labels": "cos_train_labels.csv",
    "cos_test_labels":  "cos_test_labels.csv",
}
for key, fname in file_map.items():
    between[key].to_csv(os.path.join(out_root, fname))


In [43]:
between["cos_train_words"]

,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
standard,0.000000,0.002493,0.010250,0.015372,0.012072,0.031638,0.020117
heur_len,0.002493,0.000000,0.012634,0.018225,0.014582,0.033932,0.022602
heur_rare,0.010250,0.012634,0.000000,0.016709,0.013122,0.039957,0.023860
adversarial,0.015372,0.018225,0.016709,0.000000,0.021583,0.051913,0.022633
loc,0.012072,0.014582,0.013122,0.021583,0.000000,0.039488,0.032270
semantic,0.031638,0.033932,0.039957,0.051913,0.039488,0.000000,0.050818
reverse,0.020117,0.022602,0.023860,0.022633,0.032270,0.050818,0.000000


In [44]:
between["cos_test_words"]

,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
standard,0.000000,0.102337,0.161103,0.185166,0.218516,0.320098,0.184596
heur_len,0.102337,0.000000,0.108726,0.152113,0.168419,0.265876,0.144382
heur_rare,0.161103,0.108726,0.000000,0.149234,0.147963,0.322530,0.137365
adversarial,0.185166,0.152113,0.149234,0.000000,0.239864,0.419941,0.122109
loc,0.218516,0.168419,0.147963,0.239864,0.000000,0.337993,0.229912
semantic,0.320098,0.265876,0.322530,0.419941,0.337993,0.000000,0.407597
reverse,0.184596,0.144382,0.137365,0.122109,0.229912,0.407597,0.000000


In [45]:
between["cos_train_labels"]

,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
standard,0.000000,0.000004,0.000006,0.993632,0.000012,0.000009,0.999862
heur_len,0.000004,0.000000,0.000011,0.993525,0.000016,0.000004,0.999855
heur_rare,0.000006,0.000011,0.000000,0.993660,0.000019,0.000014,0.999881
adversarial,0.993632,0.993525,0.993660,0.000000,0.993368,0.992506,0.996923
loc,0.000012,0.000016,0.000019,0.993368,0.000000,0.000020,0.999867
semantic,0.000009,0.000004,0.000014,0.992506,0.000020,0.000000,0.999839
reverse,0.999862,0.999855,0.999881,0.996923,0.999867,0.999839,0.000000


In [46]:
between["cos_test_labels"]

,standard,heur_len,heur_rare,adversarial,loc,semantic,reverse
standard,0.000000,0.000013,0.000015,0.995064,0.000044,0.000100,0.999333
heur_len,0.000013,0.000000,0.000010,0.994867,0.000033,0.000075,0.999396
heur_rare,0.000015,0.000010,0.000000,0.994797,0.000028,0.000086,0.999316
adversarial,0.995064,0.994867,0.994797,0.000000,0.995369,0.998543,0.994778
loc,0.000044,0.000033,0.000028,0.995369,0.000000,0.000050,0.999393
semantic,0.000100,0.000075,0.000086,0.998543,0.000050,0.000000,0.999528
reverse,0.999333,0.999396,0.999316,0.994778,0.999393,0.999528,0.000000
